# Pilot analysis — dependência de contexto em clarificação com LLMs

## 0. Contexto e natureza do piloto

**Este estudo é um piloto metodológico. As estatísticas apresentadas são descritivas e exploratórias e têm como objetivo avaliar a viabilidade do método e identificar padrões preliminares. Não devem ser interpretadas como estimativas generalizáveis do comportamento de LLMs ou do Spec Kit.**

Escopo deste notebook: operacionalizar o método, construir unidades analíticas, métricas descritivas e infraestrutura. Não executar Codex/Spec Kit; não alterar dados experimentais, PRR ou anotações humanas; não inferir mapping semântico pergunta→gap.


## 1. Imports e configuração

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent.parent
elif (REPO_ROOT / "analysis" / "config.py").exists():
    pass
elif (REPO_ROOT.parent / "analysis" / "config.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from analysis import config
from analysis.src.loaders import environment_versions, load_analysis_inputs
from analysis.src.validation import AnalysisValidationError, validate_all_inputs
from analysis.src.mappings import build_final_mapping, explode_question_gap_mapping
from analysis.src.matrices import build_gap_run_matrix
from analysis.src.metrics import (
    classify_repetition_stability,
    dataset_description,
    format_rate,
    gap_recall,
    occurrence_rate,
    questions_summary_stats,
    requery_rate,
    viability_metrics,
)
from analysis.src import plots
from analysis.src.statistics import (
    clustered_bootstrap_ci,
    exploratory_inferential_models_disabled,
    fit_mixed_logit_recognized_future,
)
from analysis.src.pipeline import run_pipeline

print("ANALYSIS_SEED =", config.ANALYSIS_SEED)
print("timestamp UTC:", datetime.now(timezone.utc).isoformat())
print("versions:", environment_versions())
print(config.PILOT_DISCLAIMER)


## 2–6. Carregamento, validação, mapping e Gap×Run

In [ ]:
# Pipeline completo: valida inputs → mapping final → question-gap-mapping → gap-run-matrix → tabelas/figuras.
# Se validação crítica falhar, PARA com mensagem clara (não corrige dados humanos).

try:
    artifacts = run_pipeline(pilot_asserts=True, write_outputs=True)
except AnalysisValidationError as err:
    print(str(err))
    raise SystemExit("ANÁLISE INTERROMPIDA: corrija/complete a anotação humana antes de continuar.")

print("Input hashes:")
for k, v in artifacts.meta["input_hashes"].items():
    print(f"  {k}: {v[:16]}…")

final_map = artifacts.final_map
qgm = artifacts.question_gap_mapping
matrix = artifacts.gap_run_matrix
print(f"perguntas={len(final_map)} qgm_rows={len(qgm)} gap_run_rows={len(matrix)}")
assert len(matrix) == config.PILOT_EXPECTED_GAP_RUN_ROWS
assert (matrix["recognized"].isin([0, 1])).all()
assert (matrix["question_count_for_gap"] >= matrix["recognized"]).all()
assert (matrix["gap_id"] != "NONE").all()
c0 = matrix[matrix["condition"] == "C0"]
assert (c0["gap_state"] == "OPEN").all(), "C0 deve ter todos os gaps OPEN"


## 7. Descrição do dataset

In [ ]:
desc = artifacts.meta["dataset"]
display(artifacts.tables["dataset-summary"])
display(artifacts.tables["questions-by-condition"])
print("Gaps por US:", desc["gaps_by_user_story"])
print("Gaps por categoria:", desc.get("gaps_by_category"))
print("Gaps por importance:", desc.get("gaps_by_importance"))


## 8. Viabilidade do método de anotação

Divergência entre avaliadores é reportada apenas como **diagnóstico do procedimento de anotação**.
Não se reporta Cohen's kappa como evidência de confiabilidade interavaliadores neste piloto (Avaliador 2 assistivo/ChatGPT).


In [ ]:
via = artifacts.meta["viability"]
display(artifacts.tables["viability-metrics"])
print(via["note"])
print(
    f"Acordos diretos={via['direct_agreement']}; "
    f"adjudicados={via['adjudicated']}; "
    f"burden={format_rate(via['adjudication_burden'])}; "
    f"NONE rate={format_rate(via['none_rate'])}; "
    f"multi-gap rate={format_rate(via['multi_gap_rate'])}"
)


## 9. RQ1 — baseline / C0 (somente OPEN)

Gap Recall run-level = recognized OPEN / total OPEN (ocorrências Gap×Run).
Miss Rate = 1 − Gap Recall na mesma unidade (não é descoberta independente).


In [ ]:
display(artifacts.tables["rq1-c0-global"])
display(artifacts.tables["gap-recall-c0-by-user-story"])
display(artifacts.tables["gap-recall-c0-by-importance"])
display(artifacts.tables["gap-recall-c0-by-category"])
r = artifacts.tables["rq1-c0-global"].iloc[0]["gap_recall"]
print(f"No piloto, a condição C0 apresentou Gap Recall de {format_rate(r)} (unidade Gap×Run OPEN).")


## 10. RQ2 — gaps remanescentes OPEN (CL, CO, CD, CS; CT pode ser N/A)

In [ ]:
display(artifacts.tables["gap-recall-rq2-by-condition"])
display(artifacts.tables["gap-recall-by-condition"])
# N/A nunca convertido para zero
for _, row in artifacts.tables["gap-recall-by-condition"].iterrows():
    print(f"{row['condition']}: open={row['open_gaps']} recall={format_rate(row['gap_recall'])}")


## 11. RQ3 — reconsulta de gaps ANSWERED

In [ ]:
display(artifacts.tables["requery-rate-global"])
display(artifacts.tables["requery-rate-by-condition"])
display(artifacts.tables["requery-rate-by-user-story"])
display(artifacts.tables["requery-rate-by-category"])


## 12–13. Occurrence Rate e estabilidade entre repetições (rótulos DESCRITIVOS)

In [ ]:
display(artifacts.tables["gap-occurrence-rate"].head(20))
display(artifacts.tables["repetition-stability"]["stability_label"].value_counts())
print("Rótulos 0/3…3/3 são DESCRITIVOS do piloto, não categorias universais.")


## 14. NONE / gaps não previstos (somente quantitativo)

In [ ]:
display(artifacts.tables["unmatched-summary"])
display(artifacts.tables["none-by-user-story"])
display(artifacts.tables["none-by-condition"])
print("Análise qualitativa de NONE fica para estudo exploratório separado.")


## 15. Número bruto de perguntas

**Menor quantidade de perguntas não significa necessariamente melhor clarificação.**
Não usar raw question count como métrica principal de qualidade.


In [ ]:
display(artifacts.tables["question-count-stats-per-run"])
print(artifacts.meta["question_count_warning"])


## Bootstrap clusterizado (exploratório) e modelos inferenciais desabilitados

In [ ]:
print(artifacts.meta["clustered_bootstrap_c0_recall"])
print(artifacts.meta["inferential_models"])
print(artifacts.meta["mixed_logit_future"])


## 16–18. Síntese, limitações e implicações para o experimento principal

Interpretação apenas descritiva (sem linguagem causal/confirmatória).


In [ ]:
for note in artifacts.descriptive_notes:
    print("-", note)
print()
print("Limitações: n=4 US; Avaliador 2 assistivo; CIs clusterizados tipicamente inadequados no piloto;")
print("sem testes confirmatórios automáticos; OccurrenceRate/GLMM prontos para escala futura.")
print("Implicação: completar anotação humana → reexecutar ./scripts/bin/run-analysis.sh no estudo principal com mais US.")
print("Figuras:", len(artifacts.figure_paths))
print("Tabelas:", sorted(artifacts.tables))
